# 07: BitNet demo

[microsoft/bitnet-b1.58-2B-4T](https://huggingface.co/microsoft/bitnet-b1.58-2B-4T)
(30 layers, dim=2560, 2B params). Weights stored as packed uint8 with per-layer weight_scale.

smelt.quantize() detects the ternary weights, reads weight_scale, packs into TL1 for AVX2 inference.

In [ ]:
import os
import sys
import time

if "COLAB_GPU" in os.environ:
    os.system("pip install -q git+https://github.com/PritRaj1/smelt.git")
    os.system("pip install -q 'transformers>=4.51,<5'")
else:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import smelt

## smelt.quantize

In [2]:
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
w0 = model.model.layers[0].self_attn.q_proj.weight
print(f"params: {n_params:.0f}M")
print(f"weight dtype: {w0.dtype}")
print(f"weight unique values: {sorted(w0.unique().tolist())}")
print(f"has weight_scale: {hasattr(model.model.layers[0].self_attn.q_proj, 'weight_scale')}")

You have loaded a BitNet model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model.


Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

params: 2413M
weight dtype: torch.uint8
weight unique values: [0, 1, 255]
has weight_scale: True


In [ ]:
t0 = time.perf_counter()
smelt.quantize(model)
print(f"quantized in {time.perf_counter() - t0:.1f}s")

model.generation_config.cache_implementation = "static"
model.forward = torch.compile(model.forward, fullgraph=True)

## Generate

In [ ]:
prompts = [
    "The capital of UK is",
    "def fibonacci(n):",
    "Explain x86 AVX2 in simple terms:",
]

for prompt in prompts:
    ids = tokenizer.encode(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=64, do_sample=False)

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"--- {prompt} ---")
    print(text)
    print()

## Benchmark

64 tokens, 3 runs avg.

In [ ]:
import platform
import subprocess


def get_cpu_name():
    try:
        out = subprocess.check_output(["lscpu"], text=True)
        for line in out.splitlines():
            if "Model name" in line:
                return line.split(":")[1].strip()
    except Exception:
        pass
    return platform.processor() or platform.machine()


def bench_generate(model, input_ids, n_tokens=64, warmup=2, runs=3):
    for _ in range(warmup):
        with torch.no_grad():
            model.generate(input_ids, max_new_tokens=n_tokens, do_sample=False)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(input_ids, max_new_tokens=n_tokens, do_sample=False)
        times.append(time.perf_counter() - t0)
    return n_tokens / (sum(times) / len(times))


prompt_ids = tokenizer.encode("The meaning of life is", return_tensors="pt")
cpu_name = get_cpu_name()

tps = bench_generate(model, prompt_ids)
print(f"smelt ({cpu_name}): {tps:.1f} tok/s")